# Installating Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting

# Abundance Difference

**Reading Files**

In [2]:
# Neighbour dataset - SwP with their three closest neighbours
# epsilon - Dataset with the abundance values
df1 = pd.read_csv('Neighbour Dataset.csv')
df2 = pd.read_csv('epsilon.csv')

In [3]:
df1

           SwP      First     Second      Third
0      HD21749  BD-053176   HD151692    HD20492
1     HD125595   HD214998  BD-130321  BD-002387
2    BD-082823   HD152533   HD188559    HD31560
3      HD23472    HD22897    HD50590    HD16280
4      HIP5158    HD16905   HD213042   HD214998
..         ...        ...        ...        ...
128   HD108147   HD139590   HD192865    HD75881
129   HD212301   HD206116   HD215625   HD139590
130   HD179949   HD206116    HD52449   HD139590
131   HD221287    HD68287   HD163102    HD95922
132      HD142    HD68287   HD149200   HD163102

[133 rows x 4 columns]

In [4]:
df2

          Star SwP?  [Fe/H]  e_[Fe/H]  [Fe/H]_a  e_[Fe/H]_a  [Zn/Fe]  \
0     HD160089   No    0.11      0.04      0.11        0.03    -0.05   
1     HD189567  Yes   -0.24      0.01     -0.24        0.03     0.05   
2     HD219077  Yes   -0.13      0.01     -0.13        0.03     0.04   
3     HD119638   No   -0.15      0.01     -0.15        0.03    -0.04   
4     HD140901   No    0.09      0.01      0.09        0.03    -0.01   
...        ...  ...     ...       ...       ...         ...      ...   
1020  HD105938   No    0.03      0.02      0.03        0.03    -0.13   
1021   HD37990   No    0.00      0.02      0.00        0.03    -0.09   
1022  HD149747   No   -0.34      0.03     -0.34        0.03     0.00   
1023  HD218249   No   -0.43      0.02     -0.40        0.03     0.01   
1024   HD17865   No   -0.57      0.02     -0.57        0.03     0.10   

      e_[Zn/Fe]  [Y/Fe]  e_[Y/Fe]  ...  [Sr/Fe]  e_[Sr/Fe]  [Ce/Fe]  \
0          0.06    0.01      0.09  ...     0.10       0.12     0

**Creating a dataframe with Abundance Differences**

In [5]:
l = df2.keys()[2:]

param = l[range(0,len(l),2)]
param_err = l[range(1,len(l),2)]

In [6]:
data_main = []

for i,row in df1.iterrows():
  id = row['SwP']

  if id in df2['Star'].values:
    s0 = df2[df2['Star']==id]
    data_row = [id]
    for j in ['First', 'Second','Third']:
      ng = row[j]
      data_row += [ng]
      if ng in df2['Star'].values:

        s = df2[df2['Star'] == ng]
        p = (s0[param].values[0] - s[param].values[0])
        p_err = np.sqrt( s0[param_err].values[0]**2 + s[param_err].values[0]**2 )

        p = list(p)
        p_err = list(p_err)

        data_row += p + p_err

      else:
        data_row += [np.nan]*48
    data_main.append(data_row)

In [7]:
columns = ['id']
ids = ['First', 'Second','Third']
for i in range(1,4):
  columns+= [ids[i-1]]
  c = [f'{i}_'+ j for j in param]
  c_err = [f'{i}_'+ j for j in param_err]

  columns += c + c_err

In [8]:
df3 = pd.DataFrame(data_main, columns=columns)

In [9]:
df3.to_csv('Abundance Difference.csv')

In [10]:
df3

            id      First  1_[Fe/H]  1_[Fe/H]_a  1_[Zn/Fe]  1_[Y/Fe]  \
0      HD21749  BD-053176      0.01        0.07       0.01      0.08   
1     HD125595   HD214998      0.00        0.02      -0.02      0.00   
2    BD-082823   HD152533      0.01       -0.02      -0.04      0.02   
3      HD23472    HD22897      0.01        0.06       0.06     -0.11   
4      HIP5158    HD16905      0.02        0.02      -0.08      0.02   
..         ...        ...       ...         ...        ...       ...   
128   HD108147   HD139590      0.05        0.05      -0.11      0.01   
129   HD212301   HD206116     -0.06       -0.06      -0.02      0.01   
130   HD179949   HD206116     -0.03       -0.03      -0.01      0.03   
131   HD221287    HD68287     -0.02       -0.02      -0.02      0.10   
132      HD142    HD68287      0.03        0.03       0.03      0.13   

     1_[Sc/Fe]  1_[Mg/Fe]  1_[Ca/Fe]  1_[Na/Fe]  ...  3_e_[Mn/Fe]  \
0       -0.062      -0.03      -0.02     -0.065  ...     0.207183 

In [11]:
# dropping the rows with Nan values
df_f = df3.dropna()
df_f

            id      First  1_[Fe/H]  1_[Fe/H]_a  1_[Zn/Fe]  1_[Y/Fe]  \
0      HD21749  BD-053176      0.01        0.07       0.01      0.08   
1     HD125595   HD214998      0.00        0.02      -0.02      0.00   
2    BD-082823   HD152533      0.01       -0.02      -0.04      0.02   
3      HD23472    HD22897      0.01        0.06       0.06     -0.11   
4      HIP5158    HD16905      0.02        0.02      -0.08      0.02   
..         ...        ...       ...         ...        ...       ...   
127    HD17051    HD66039      0.02        0.02       0.09     -0.11   
129   HD212301   HD206116     -0.06       -0.06      -0.02      0.01   
130   HD179949   HD206116     -0.03       -0.03      -0.01      0.03   
131   HD221287    HD68287     -0.02       -0.02      -0.02      0.10   
132      HD142    HD68287      0.03        0.03       0.03      0.13   

     1_[Sc/Fe]  1_[Mg/Fe]  1_[Ca/Fe]  1_[Na/Fe]  ...  3_e_[Mn/Fe]  \
0       -0.062      -0.03      -0.02     -0.065  ...     0.207183 

In [12]:
df_f.to_csv('final_test.csv')

**General Properties of the Stars**

In [13]:
# General Properties - Dataset of stars with T_eff, log_g, Metallicity and RA-Dec
df4 = pd.read_csv('General Properties.csv')
df4

            Star  Teff (K)  e_Teff  logg [cm/s2]  e_logg  [Fe/H]  e_[Fe/H]  \
0       HIP38324      4393    98.0          4.01    0.39   -0.32      0.03   
1        HD85512      4400    45.0          4.36    0.10   -0.26      0.14   
2       HD192961      4418    51.0          4.23    0.18   -0.30      0.22   
3       HD218511      4424    90.0          4.52    0.26   -0.05      0.02   
4       HIP25612      4433    95.1          4.33    0.35   -0.49      0.06   
...          ...       ...     ...           ...     ...     ...       ...   
1023     HD66740      6666    39.0          4.49    0.06    0.04      0.02   
1024    HD141128      6758    48.0          4.67    0.06    0.07      0.03   
1025  CD-2310879      6788    43.0          4.67    0.04   -0.24      0.02   
1026     HD20852      6813    70.0          4.76    0.06   -0.35      0.04   
1027     HD38385      7212    83.0          4.61    0.11    0.09      0.05   

      RA (degree)  Dec (degree)  
0       117.76394     -50.305

In [14]:
df4 = df4[['Star','Teff (K)','e_Teff','logg [cm/s2]','e_logg','RA (degree)','Dec (degree)','[Fe/H]',	'e_[Fe/H]']]
df4

            Star  Teff (K)  e_Teff  logg [cm/s2]  e_logg  RA (degree)  \
0       HIP38324      4393    98.0          4.01    0.39    117.76394   
1        HD85512      4400    45.0          4.36    0.10    147.77938   
2       HD192961      4418    51.0          4.23    0.18    304.90390   
3       HD218511      4424    90.0          4.52    0.26    347.42063   
4       HIP25612      4433    95.1          4.33    0.35     82.07632   
...          ...       ...     ...           ...     ...          ...   
1023     HD66740      6666    39.0          4.49    0.06    121.00545   
1024    HD141128      6758    48.0          4.67    0.06    236.90875   
1025  CD-2310879      6788    43.0          4.67    0.04    195.86471   
1026     HD20852      6813    70.0          4.76    0.06     50.18577   
1027     HD38385      7212    83.0          4.61    0.11     85.87577   

      Dec (degree)  [Fe/H]  e_[Fe/H]  
0        -50.30500   -0.32      0.03  
1        -43.50278   -0.26      0.14  
2     

In [15]:
df4.to_csv('Properties_f.csv')

In [16]:
df5 = pd.merge(df_f, df4, left_on='id', right_on='Star', how='inner')
df5 = df5.drop(['Star'], axis=1)
df5

            id      First  1_[Fe/H]  1_[Fe/H]_a  1_[Zn/Fe]  1_[Y/Fe]  \
0      HD21749  BD-053176      0.01        0.07       0.01      0.08   
1     HD125595   HD214998      0.00        0.02      -0.02      0.00   
2    BD-082823   HD152533      0.01       -0.02      -0.04      0.02   
3      HD23472    HD22897      0.01        0.06       0.06     -0.11   
4      HIP5158    HD16905      0.02        0.02      -0.08      0.02   
..         ...        ...       ...         ...        ...       ...   
126    HD17051    HD66039      0.02        0.02       0.09     -0.11   
127   HD212301   HD206116     -0.06       -0.06      -0.02      0.01   
128   HD179949   HD206116     -0.03       -0.03      -0.01      0.03   
129   HD221287    HD68287     -0.02       -0.02      -0.02      0.10   
130      HD142    HD68287      0.03        0.03       0.03      0.13   

     1_[Sc/Fe]  1_[Mg/Fe]  1_[Ca/Fe]  1_[Na/Fe]  ...  3_e_[Cr/Fe]  \
0       -0.062      -0.03      -0.02     -0.065  ...     0.232398 

In [17]:
df5 = pd.read_csv('Final Dataset_f.csv') #crossmatched with Abundance diff and General Properties dataframes using Topcat
df5

     col1_1       Star  Teff (K)  e_Teff  logg [cm/s2]  e_logg  RA (degree)  \
0        29    HD21749      4562   120.0          4.24    0.34     51.74676   
1        64   HD125595      4636    83.9          4.20    0.26    215.34661   
2        81  BD-082823      4672   104.5          4.33    0.26    150.19884   
3        87    HD23472      4683    49.3          4.32    0.16     55.45999   
4        98    HIP5158      4708   119.1          4.22    0.33     16.50854   
..      ...        ...       ...     ...           ...     ...          ...   
126     967    HD17051      6227    26.0          4.53    0.06     40.63944   
127     982   HD212301      6271    26.0          4.55    0.03    336.87883   
128     988   HD179949      6287    28.0          4.54    0.04    288.88846   
129    1007   HD221287      6374    29.0          4.62    0.03    352.83474   
130    1009      HD142      6403    65.0          4.62    0.07      1.57990   

     Dec (degree)  [Fe/H]  e_[Fe/H]  ...  3_e_[Mn/F

In [18]:
df5 = df5.drop(['col1_1','col1_2'], axis=1)
df5

          Star  Teff (K)  e_Teff  logg [cm/s2]  e_logg  RA (degree)  \
0      HD21749      4562   120.0          4.24    0.34     51.74676   
1     HD125595      4636    83.9          4.20    0.26    215.34661   
2    BD-082823      4672   104.5          4.33    0.26    150.19884   
3      HD23472      4683    49.3          4.32    0.16     55.45999   
4      HIP5158      4708   119.1          4.22    0.33     16.50854   
..         ...       ...     ...           ...     ...          ...   
126    HD17051      6227    26.0          4.53    0.06     40.63944   
127   HD212301      6271    26.0          4.55    0.03    336.87883   
128   HD179949      6287    28.0          4.54    0.04    288.88846   
129   HD221287      6374    29.0          4.62    0.03    352.83474   
130      HD142      6403    65.0          4.62    0.07      1.57990   

     Dec (degree)  [Fe/H]  e_[Fe/H]         id  ... 3_e_[Mn/Fe]  3_e_[Ti/Fe]  \
0       -63.49910    0.02      0.07    HD21749  ...    0.207183    

**Extracting Abundance Difference for each SwP with it's first, second and third neighbour**

In [19]:
# List of elements and their Tc
element_list = {
    'Cu': 1034,
    'Na': 1035,
    'Mn': 1123,
    'Cr': 1291,
    'Si': 1314,
    'Mg': 1343,
    'Co': 1354,
    'Ni': 1363,
    'V':  1370,
    'Ba': 1423,
    'Ce': 1454,
    'Ca': 1535,
    'Sc': 1541,
    'Sr': 1548,
    'Y' : 1551,
    'Ti': 1565,
    'Al': 1652,

}

def extract_abundance_diff(star_name):
    # Find row number for given star name
    row = df_f.loc[df_f['id'] == star_name]
    row_index = row.index[0]

    # Extract column values for given elements
    abundance_diff1 = []
    abundance_diff2 = []
    abundance_diff3 = []

    error = []
    for element in element_list.keys():
        col_name = f'1_[{element}/Fe]'
        error_col_name = f'1_e_[{element}/Fe]'
        if col_name in df_f.columns and error_col_name in df_f.columns:
            col_value = df_f.at[row_index, col_name]
            #error_value = df5.at[row_index, error_col_name]
            abundance_diff1.append(col_value)
           # error.append(error_value)

    for element in element_list.keys():
        col_name = f'2_[{element}/Fe]'
        error_col_name = f'2_e_[{element}/Fe]'
        if col_name in df_f.columns and error_col_name in df_f.columns:
            col_value = df_f.at[row_index, col_name]
            #error_value = df5.at[row_index, error_col_name]
            abundance_diff2.append(col_value)
           # error.append(error_value)

    for element in element_list.keys():
        col_name = f'3_[{element}/Fe]'
        error_col_name = f'3_e_[{element}/Fe]'
        if col_name in df_f.columns and error_col_name in df_f.columns:
            col_value = df_f.at[row_index, col_name]
            #error_value = df5.at[row_index, error_col_name]
            abundance_diff3.append(col_value)
           # error.append(error_value)

    return abundance_diff1,abundance_diff2, abundance_diff3  #error]

In [20]:
df = pd.read_csv('Condensation Temps.csv')
df_T = df.drop(df.index[[0,2,8,15,22,23,24]])
df_T


   Element  Temperature
1       Al         1652
3       Ti         1565
4        Y         1551
5       Sr         1548
6       Sc         1541
7       Ca         1535
9       Ce         1454
10      Ba         1423
11       V         1370
12      Ni         1363
13      Co         1354
14      Mg         1343
16      Si         1314
17      Cr         1291
18      Mn         1123
19      Na         1035
20      Cu         1034
21      Zn          704

In [21]:
ext = extract_abundance_diff('HD21749')
ext[0]

[0.009999999999999995,
 -0.065,
 -0.002,
 0.035,
 -0.03,
 -0.03,
 -0.027999999999999997,
 -0.0049999999999999975,
 0.06900000000000006,
 0.03,
 -0.06,
 -0.020000000000000004,
 -0.062,
 0.010000000000000009,
 0.08,
 -0.020000000000000004,
 0.0]

**Model Fitting and Slope Estimation**

In [22]:
def fit_line(x,y):
  init = models.Linear1D()
  fit  = fitting.LinearLSQFitter()
  model = fit(init,x,y)
  return model

In [23]:
from scipy.stats import linregress

df_T = df_T[df_T['Element'].isin(element_list.keys())].reset_index(drop=True)

# Helper function: fits a line with sigma clipping and returns slope, intercept, and p-value
def fit_line_with_stats(x, y):
    x = np.array(x)
    y = np.array(y)
    
    # Remove NaNs
    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]
    
    # Sigma clipping: 2σ
    mean = np.mean(y)
    std = np.std(y)
    mask = np.abs(y - mean) < 3 * std
    x = x[mask]
    y = y[mask]

    # Fit the line
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    return slope, intercept, p_value, x, y

# Main loop
dat = []
for i, row in df_f.iterrows():
    star_id = row['id']
    ext = extract_abundance_diff(star_id)

    l0 = np.array(ext[0]).flatten()
    l1 = np.array(ext[1]).flatten()
    l2 = np.array(ext[2]).flatten()

    T = df_T['Temperature'].values

    slope1, intercept1, pval1, x1, y1 = fit_line_with_stats(T, l0)
    slope2, intercept2, pval2, x2, y2 = fit_line_with_stats(T, l1)
    slope3, intercept3, pval3, x3, y3 = fit_line_with_stats(T, l2)

    dat.append([star_id, slope1, slope2, slope3, pval1, pval2, pval3])

    # Plotting
    fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(30, 8))
    
    ax[0].scatter(x1, y1)
    ax[0].plot(x1, intercept1 + slope1 * x1, '--', color='r')
    ax[0].set_title("1st Neighbor")

    ax[1].scatter(x2, y2)
    ax[1].plot(x2, intercept2 + slope2 * x2, '--', color='r')
    ax[1].set_title("2nd Neighbor")

    ax[2].scatter(x3, y3)
    ax[2].plot(x3, intercept3 + slope3 * x3, '--', color='r')
    ax[2].set_title("3rd Neighbor")

    fig.suptitle(f"SwP: {star_id}", y=0.97, x=0.50)
    plt.show()




<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

...

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

In [ ]:
df = pd.DataFrame(dat, columns=['Star','slope1','slope2','slope3','pval1','pval2','pval3'])

# Zero out slopes with p > 0.05
for i in range(1, 4):
    df.loc[df[f'pval{i}'] > 0.05, f'slope{i}'] = 0


In [ ]:

df_neg = df[(df['slope1'] < 0) & (df['slope2'] < 0) & (df['slope3'] < 0)]

df_pos = df[(df['slope1'] > 0) & (df['slope2'] > 0) & (df['slope3'] > 0)]

In [32]:
#Function for assigning values for stars with all negative slopes and all positive slopes

def sign_of_slopes(row):
    all_positive = all(slope > 0 for slope in row)
    all_negative = all(slope < 0 for slope in row)

    if all_positive:
        return 1
    elif all_negative:
        return -1
    else:
        return 0

# Apply the function row-wise to create the new column
df['assigned_val'] = df[['slope1', 'slope2', 'slope3']].apply(sign_of_slopes, axis=1)

df

          Star    slope1    slope2  slope3     pval1     pval2     pval3  \
0      HD21749  0.000000 -0.000100     0.0  0.962455  0.022732  0.246623   
1     HD125595  0.000000  0.000000     0.0  0.866865  0.607413  0.633017   
2    BD-082823  0.000000  0.000000     0.0  0.775099  0.917593  0.211752   
3      HD23472  0.000000  0.000000     0.0  0.322157  0.749934  0.055574   
4      HIP5158  0.000000  0.000000     0.0  0.254890  0.444250  0.499787   
..         ...       ...       ...     ...       ...       ...       ...   
126    HD17051  0.000115  0.000000     0.0  0.043737  0.166890  0.819189   
127   HD212301  0.000000  0.000121     0.0  0.970036  0.038728  0.802300   
128   HD179949 -0.000170  0.000000     0.0  0.044732  0.275008  0.365052   
129   HD221287  0.000000  0.000000     0.0  0.275076  0.368604  0.341442   
130      HD142  0.000000  0.000000     0.0  0.388975  0.093321  0.891916   

     assigned_val  
0               0  
1               0  
2               0  
3      

In [33]:
# Classify based on number of significant positive/negative slopes
def classify_partial_slopes(row):
    slopes = [row['slope1'], row['slope2'], row['slope3']]
    signs = [np.sign(s) for s in slopes if s != 0]  # Ignore zero slopes (insignificant)
    pos = signs.count(1)
    neg = signs.count(-1)

    if pos == 3:
        return 'all_pos'
    elif neg == 3:
        return 'all_neg'
    elif pos == 2:
        return '2_pos'
    elif neg == 2:
        return '2_neg'
    else:
        return 'mixed_or_zero'

# Apply the function and count each category
df['slope_pattern'] = df.apply(classify_partial_slopes, axis=1)
print(df['slope_pattern'].value_counts())  # Show how many fall into each group


slope_pattern
mixed_or_zero    124
2_neg              4
2_pos              2
all_neg            1
Name: count, dtype: int64


In [37]:
df_neg

          Star    slope1        slope2    slope3
0      HD21749 -0.000029 -7.826191e-05 -0.000176
1     HD125595 -0.000058 -3.098567e-05 -0.000039
4      HIP5158 -0.000105 -1.363379e-05 -0.000042
7      HD63454 -0.000051 -4.014995e-05 -0.000130
12      HD8326 -0.000012 -8.792300e-05 -0.000072
16    HD330075 -0.000158 -3.492378e-06 -0.000025
36     HD73267 -0.000060 -9.071299e-05 -0.000021
42     HD69830 -0.000023 -2.056098e-05 -0.000081
46     HD63765 -0.000006 -7.147988e-05 -0.000050
49    HD111232 -0.000028 -1.858521e-05 -0.000016
54   HD142022A -0.000009 -4.436793e-05 -0.000007
57     HD73256 -0.000118 -8.467886e-05 -0.000054
60    HD168746 -0.000065 -3.032048e-05 -0.000071
70    HD102117 -0.000007 -8.314382e-05 -0.000079
80    HD189567 -0.000010 -1.938385e-05 -0.000002
83     HD92788 -0.000026 -8.723222e-05 -0.000022
90    HD109271 -0.000036 -5.688750e-05 -0.000061
94     HD16141 -0.000025 -5.440866e-05 -0.000047
95     HD16417 -0.000048 -2.580933e-05 -0.000007
97    HD147513 -0.00

In [38]:
df_pos


         Star    slope1        slope2        slope3
3     HD23472  0.000058  2.731161e-05  9.055372e-05
9     HD40307  0.000171  1.578513e-04  1.815484e-05
10   HD215152  0.000048  6.356293e-05  1.970487e-04
11    HD27894  0.000033  1.184323e-05  9.062630e-05
13   HD104067  0.000008  9.158136e-08  1.486462e-05
21    HD22049  0.000008  6.991617e-05  6.815542e-05
22   HD114783  0.000142  1.119652e-04  1.553193e-04
28    HD85390  0.000033  1.090334e-04  8.623472e-05
34    HD51608  0.000111  3.960069e-06  4.457612e-05
53   HD290327  0.000056  2.680057e-05  3.796588e-05
55    HD83443  0.000013  5.882026e-05  2.804096e-05
61     HD4208  0.000042  6.013232e-05  1.726815e-05
62    HD90156  0.000042  3.941616e-05  2.891127e-05
65   HD177565  0.000021  1.281047e-05  6.883961e-07
68   HD126525  0.000063  3.941753e-05  1.158963e-04
75    HD70642  0.000071  9.466168e-05  6.073153e-05
76    HD47186  0.000017  2.768431e-05  1.951377e-05
77    HD66428  0.000082  5.996588e-06  3.860445e-05
79     HD671

In [39]:
df_neg=df[df['assigned_val']==-1]
df_neg

          Star    slope1        slope2    slope3  assigned_val
0      HD21749 -0.000029 -7.826191e-05 -0.000176            -1
1     HD125595 -0.000058 -3.098567e-05 -0.000039            -1
4      HIP5158 -0.000105 -1.363379e-05 -0.000042            -1
7      HD63454 -0.000051 -4.014995e-05 -0.000130            -1
12      HD8326 -0.000012 -8.792300e-05 -0.000072            -1
16    HD330075 -0.000158 -3.492378e-06 -0.000025            -1
36     HD73267 -0.000060 -9.071299e-05 -0.000021            -1
42     HD69830 -0.000023 -2.056098e-05 -0.000081            -1
46     HD63765 -0.000006 -7.147988e-05 -0.000050            -1
49    HD111232 -0.000028 -1.858521e-05 -0.000016            -1
54   HD142022A -0.000009 -4.436793e-05 -0.000007            -1
57     HD73256 -0.000118 -8.467886e-05 -0.000054            -1
60    HD168746 -0.000065 -3.032048e-05 -0.000071            -1
70    HD102117 -0.000007 -8.314382e-05 -0.000079            -1
80    HD189567 -0.000010 -1.938385e-05 -0.000002       

In [40]:
df_pos = df[df['assigned_val']==1]
df_pos

         Star    slope1        slope2        slope3  assigned_val
3     HD23472  0.000058  2.731161e-05  9.055372e-05             1
9     HD40307  0.000171  1.578513e-04  1.815484e-05             1
10   HD215152  0.000048  6.356293e-05  1.970487e-04             1
11    HD27894  0.000033  1.184323e-05  9.062630e-05             1
13   HD104067  0.000008  9.158136e-08  1.486462e-05             1
21    HD22049  0.000008  6.991617e-05  6.815542e-05             1
22   HD114783  0.000142  1.119652e-04  1.553193e-04             1
28    HD85390  0.000033  1.090334e-04  8.623472e-05             1
34    HD51608  0.000111  3.960069e-06  4.457612e-05             1
53   HD290327  0.000056  2.680057e-05  3.796588e-05             1
55    HD83443  0.000013  5.882026e-05  2.804096e-05             1
61     HD4208  0.000042  6.013232e-05  1.726815e-05             1
62    HD90156  0.000042  3.941616e-05  2.891127e-05             1
65   HD177565  0.000021  1.281047e-05  6.883961e-07             1
68   HD126

In [42]:

# Function to assign values if two or more slopes are positive or negative
df=pd.DataFrame(dat, columns=['Star','slope1','slope2','slope3'])

def assign_value(row):
    row_values = row[1:]  # Exclude the first column ('Names')
    positives = np.sum(row_values > 0)
    negatives = np.sum(row_values < 0)
    if positives >= 2:
        return 1
    elif negatives >= 2:
        return -1
    else:
        return 0

# Apply the function to each row and assign the result to a new column
df['assigned_value'] = df.apply(assign_value, axis=1)

df


          Star    slope1    slope2    slope3  assigned_value
0      HD21749 -0.000029 -0.000078 -0.000176              -1
1     HD125595 -0.000058 -0.000031 -0.000039              -1
2    BD-082823  0.000004 -0.000040  0.000013               1
3      HD23472  0.000058  0.000027  0.000091               1
4      HIP5158 -0.000105 -0.000014 -0.000042              -1
..         ...       ...       ...       ...             ...
126    HD17051  0.000131 -0.000132 -0.000023              -1
127   HD212301 -0.000048  0.000120  0.000061               1
128   HD179949 -0.000130  0.000062 -0.000022              -1
129   HD221287 -0.000066 -0.000061 -0.000187              -1
130      HD142  0.000097  0.000269  0.000101               1

[131 rows x 5 columns]

In [43]:
df_neg = df[(df['assigned_value']==-1)]
df_neg

         Star    slope1    slope2    slope3  assigned_value
0     HD21749 -0.000029 -0.000078 -0.000176              -1
1    HD125595 -0.000058 -0.000031 -0.000039              -1
4     HIP5158 -0.000105 -0.000014 -0.000042              -1
6     HD20868  0.000004 -0.000098 -0.000098              -1
7     HD63454 -0.000051 -0.000040 -0.000130              -1
..        ...       ...       ...       ...             ...
123  HD208487  0.000040 -0.000076 -0.000071              -1
124    HD8535 -0.000109  0.000020 -0.000068              -1
126   HD17051  0.000131 -0.000132 -0.000023              -1
128  HD179949 -0.000130  0.000062 -0.000022              -1
129  HD221287 -0.000066 -0.000061 -0.000187              -1

[67 rows x 5 columns]

In [44]:
df_pos = df[(df['assigned_value']==1)]
df_pos

          Star    slope1    slope2    slope3  assigned_value
2    BD-082823  0.000004 -0.000040  0.000013               1
3      HD23472  0.000058  0.000027  0.000091               1
5     HD162020 -0.000034  0.000105  0.000077               1
8     HD114386 -0.000042  0.000027  0.000068               1
9      HD40307  0.000171  0.000158  0.000018               1
..         ...       ...       ...       ...             ...
120    HD24085  0.000098  0.000167  0.000061               1
122   HD209458  0.000022  0.000046  0.000026               1
125    HD25171  0.000088  0.000034  0.000059               1
127   HD212301 -0.000048  0.000120  0.000061               1
130      HD142  0.000097  0.000269  0.000101               1

[64 rows x 5 columns]

In [46]:
df_comp = pd.read_csv('Composite_data.csv')
df_comp

         pl_name  hostname    hd_name  sy_snum  sy_pnum  sy_mnum  \
0       11 Com b    11 Com  HD 107383        2        1        0   
1       11 UMi b    11 UMi  HD 136726        1        1        0   
2       14 And b    14 And  HD 221345        1        1        0   
3       14 Her b    14 Her  HD 145675        1        2        0   
4     16 Cyg B b  16 Cyg B  HD 186427        3        1        0   
...          ...       ...        ...      ...      ...      ...   
5590   ups And b   ups And    HD 9826        2        3        0   
5591   ups And c   ups And    HD 9826        2        3        0   
5592   ups And d   ups And    HD 9826        2        3        0   
5593   ups Leo b   ups Leo        NaN        1        1        0   
5594    xi Aql b    xi Aql  HD 188310        1        1        0   

      discoverymethod  pl_controv_flag    pl_orbper  pl_orbpererr1  ...  \
0     Radial Velocity                0   323.210000       0.060000  ...   
1     Radial Velocity            

In [53]:
df_comp['hd_name'] = df_comp['hd_name'].str.replace(' ','')
df_comp

         pl_name  hostname   hd_name  sy_snum  sy_pnum  sy_mnum  \
0       11 Com b    11 Com  HD107383        2        1        0   
1       11 UMi b    11 UMi  HD136726        1        1        0   
2       14 And b    14 And  HD221345        1        1        0   
3       14 Her b    14 Her  HD145675        1        2        0   
4     16 Cyg B b  16 Cyg B  HD186427        3        1        0   
...          ...       ...       ...      ...      ...      ...   
5590   ups And b   ups And    HD9826        2        3        0   
5591   ups And c   ups And    HD9826        2        3        0   
5592   ups And d   ups And    HD9826        2        3        0   
5593   ups Leo b   ups Leo       NaN        1        1        0   
5594    xi Aql b    xi Aql  HD188310        1        1        0   

      discoverymethod  pl_controv_flag    pl_orbper  pl_orbpererr1  ...  \
0     Radial Velocity                0   323.210000       0.060000  ...   
1     Radial Velocity                0   516.

In [54]:
merged_df = pd.merge(df_comp, df, left_on='hd_name', right_on='Star', how='inner')
merged_df

        pl_name hostname   hd_name  sy_snum  sy_pnum  sy_mnum  \
0      61 Vir b   61 Vir  HD115617        1        3        0   
1      61 Vir c   61 Vir  HD115617        1        3        0   
2      61 Vir d   61 Vir  HD115617        1        3        0   
3      GJ 143 b   GJ 143   HD21749        1        2        0   
4    HD 21749 c   GJ 143   HD21749        1        2        0   
..          ...      ...       ...      ...      ...      ...   
199   eps Eri b  eps Eri   HD22049        1        1        0   
200   tau Cet e  tau Cet   HD10700        1        4        0   
201   tau Cet f  tau Cet   HD10700        1        4        0   
202   tau Cet g  tau Cet   HD10700        1        4        0   
203   tau Cet h  tau Cet   HD10700        1        4        0   

     discoverymethod  pl_controv_flag   pl_orbper  pl_orbpererr1  ...  \
0    Radial Velocity                0     4.21500        0.00060  ...   
1    Radial Velocity                0    38.02100        0.03400  ...   


In [55]:
merged_df1 = pd.merge(df, df_comp, left_on='Star', right_on='hd_name', how='inner')
merged_df1

         Star    slope1    slope2    slope3  assigned_value      pl_name  \
0     HD21749 -0.000029 -0.000078 -0.000176              -1     GJ 143 b   
1     HD21749 -0.000029 -0.000078 -0.000176              -1   HD 21749 c   
2    HD125595 -0.000058 -0.000031 -0.000039              -1  HD 125595 b   
3     HD23472  0.000058  0.000027  0.000091               1   HD 23472 b   
4     HD23472  0.000058  0.000027  0.000091               1   HD 23472 c   
..        ...       ...       ...       ...             ...          ...   
199  HD179949 -0.000130  0.000062 -0.000022              -1  HD 179949 b   
200  HD221287 -0.000066 -0.000061 -0.000187              -1  HD 221287 b   
201     HD142  0.000097  0.000269  0.000101               1   HD 142 A d   
202     HD142  0.000097  0.000269  0.000101               1     HD 142 b   
203     HD142  0.000097  0.000269  0.000101               1     HD 142 c   

      hostname   hd_name  sy_snum  sy_pnum  ...         rastr          ra  \
0       GJ

In [56]:
merged_df['pl_bmassj']

0      0.01600
1      0.05700
2      0.07200
3      0.07142
4      0.01164
        ...   
199    0.66000
200    0.01237
201    0.01237
202    0.00551
203    0.00576
Name: pl_bmassj, Length: 204, dtype: float64

In [57]:
small_pl = merged_df[merged_df['pl_bmassj']<0.3]
giant_pl = merged_df[merged_df['pl_bmassj']>0.3]

In [58]:
len(small_pl[small_pl['assigned_value']>0])

63

In [59]:
pos_s =  len(small_pl[small_pl['assigned_value']>0])
neg_s = len(small_pl[small_pl['assigned_value']<0])
pos_g = len(giant_pl[giant_pl['assigned_value']>0])
neg_g = len(giant_pl[giant_pl['assigned_value']<0])

In [60]:
pos_g

44

In [61]:
neg_g

45

In [62]:
pos_s

63

In [63]:
neg_s

52

In [64]:
plt.hist(small_pl['assigned_value'], alpha=0.5, label='Small Planets')
plt.hist(giant_pl['assigned_value'], alpha=0.5, label='Giant Planets')
plt.legend()

<Figure size 640x480 with 1 Axes>

In [65]:
labels = 'SP_pos', 'SP_neg', 'GP_pos', 'GP_neg'
sizes = [pos_s, neg_s, pos_g, neg_g]

fig, ax = plt.subplots()
ax.pie(sizes, labels=labels,autopct='%.0f%%')


([<matplotlib.patches.Wedge at 0x7dc6111285b0>,
 [Text(0.6216500658608537, 0.9074972152107116, 'SP_pos'),
  Text(-1.0129960799721887, 0.42876443644614387, 'SP_neg'),
  Text(-0.520402838686271, -0.9691134533620256, 'GP_pos'),
  Text(0.8462674300053707, -0.7027314116446661, 'GP_neg')],
 [Text(0.3390818541059201, 0.49499848102402444, '31%'),
  Text(-0.5525433163484665, 0.2338715107888057, '25%'),
  Text(-0.28385609382887506, -0.5286073381974684, '22%'),
  Text(0.4616004163665658, -0.38330804271527236, '22%')])

<Figure size 640x480 with 1 Axes>

# Miscellaneous Section

In [66]:
#check slopes again
#small and giant planet demarcation
#account for missing planets
# multiplaanetary systems
#pie chart

In [67]:
small_pl

        pl_name  hostname   hd_name  sy_snum  sy_pnum  sy_mnum  \
0      61 Vir b    61 Vir  HD115617        1        3        0   
1      61 Vir c    61 Vir  HD115617        1        3        0   
2      61 Vir d    61 Vir  HD115617        1        3        0   
3      GJ 143 b    GJ 143   HD21749        1        2        0   
4    HD 21749 c    GJ 143   HD21749        1        2        0   
..          ...       ...       ...      ...      ...      ...   
197  HD 96700 d  HD 96700   HD96700        1        3        0   
200   tau Cet e   tau Cet   HD10700        1        4        0   
201   tau Cet f   tau Cet   HD10700        1        4        0   
202   tau Cet g   tau Cet   HD10700        1        4        0   
203   tau Cet h   tau Cet   HD10700        1        4        0   

     discoverymethod  pl_controv_flag  pl_orbper  pl_orbpererr1  ...  \
0    Radial Velocity                0    4.21500        0.00060  ...   
1    Radial Velocity                0   38.02100        0.03400

In [68]:
giant_pl[giant_pl['assigned_value']==-1]

           pl_name     hostname    hd_name  sy_snum  sy_pnum  sy_mnum  \
20     HD 108341 b    HD 108341   HD108341        2        1        0   
23     HD 111232 b    HD 111232   HD111232        1        2        0   
24     HD 111232 c    HD 111232   HD111232        1        2        0   
28     HD 117207 b    HD 117207   HD117207        1        1        0   
30     HD 121504 b    HD 121504   HD121504        1        1        0   
32     HD 125612 b    HD 125612   HD125612        2        3        0   
34     HD 125612 d    HD 125612   HD125612        2        3        0   
36     HD 130322 b    HD 130322   HD130322        1        1        0   
44     HD 134987 b    HD 134987   HD134987        1        2        0   
45     HD 134987 c    HD 134987   HD134987        1        2        0   
52     HD 141937 b    HD 141937   HD141937        1        1        0   
56   HD 142022 A b  HD 142022 A  HD142022A        2        1        0   
57     HD 145377 b    HD 145377   HD145377        1

In [69]:
df_f = pd.read_csv('Final_final.csv')
df_f

     col1_1      pl_name   hostname   hd_name  sy_snum  sy_pnum  sy_mnum  \
0        33     61 Vir b     61 Vir  HD115617        1        3        0   
1       193     GJ 143 b     GJ 143   HD21749        1        2        0   
2       213    GJ 3021 b    GJ 3021    HD1237        2        1        0   
3       469  HD 100777 b  HD 100777  HD100777        1        1        0   
4       470   HD 10180 c   HD 10180   HD10180        1        6        0   
..      ...          ...        ...       ...      ...      ...      ...   
116    1207   HD 93385 b   HD 93385   HD93385        1        3        0   
117    1227   HD 96700 b   HD 96700   HD96700        1        3        0   
118    1319     HR 810 b     HR 810   HD17051        1        1        0   
119    5557    eps Eri b    eps Eri   HD22049        1        1        0   
120    5585    tau Cet e    tau Cet   HD10700        1        4        0   

     discoverymethod  pl_controv_flag   pl_orbper  ...  sy_disterr2  \
0    Radial Velo

In [71]:
df_negf = merged_df[(merged_df['assigned_value']==-1)]
df_posf = merged_df[(merged_df['assigned_value']==1)]

In [72]:
df_posf

         pl_name   hostname   hd_name  sy_snum  sy_pnum  sy_mnum  \
5      GJ 3021 b    GJ 3021    HD1237        2        1        0   
6    HD 100777 b  HD 100777  HD100777        1        1        0   
7     HD 10180 c   HD 10180   HD10180        1        6        0   
8     HD 10180 d   HD 10180   HD10180        1        6        0   
9     HD 10180 e   HD 10180   HD10180        1        6        0   
..           ...        ...       ...      ...      ...      ...   
188   HD 90156 b   HD 90156   HD90156        1        1        0   
195   HD 96700 b   HD 96700   HD96700        1        3        0   
196   HD 96700 c   HD 96700   HD96700        1        3        0   
197   HD 96700 d   HD 96700   HD96700        1        3        0   
199    eps Eri b    eps Eri   HD22049        1        1        0   

     discoverymethod  pl_controv_flag   pl_orbper  pl_orbpererr1  ...  \
5    Radial Velocity                0   133.71000        0.20000  ...   
6    Radial Velocity                0

In [73]:
plt.scatter(df_negf['pl_bmassj'],df_negf['pl_radj'],c='b')
plt.scatter(df_posf['pl_bmassj'],df_posf['pl_radj'],c='r')

<Figure size 640x480 with 1 Axes>

In [74]:
df_comp.to_csv('Comp_data.csv')

In [76]:
df_neg = df_f[(df_f['assigned_value']==-1)]
df_pos = df_f[(df_f['assigned_value']==1)]


In [77]:
#plt.scatter(df_neg['pl_bmassj'],df_neg['pl_radj'],c='b')
plt.scatter(df_pos['pl_bmassj'],df_pos['pl_radj'],c='r')

<Figure size 640x480 with 1 Axes>

In [ ]:
df_comp

         pl_name  hostname   hd_name  sy_snum  sy_pnum  sy_mnum  \
0       11 Com b    11 Com  HD107383        2        1        0   
1       11 UMi b    11 UMi  HD136726        1        1        0   
2       14 And b    14 And  HD221345        1        1        0   
3       14 Her b    14 Her  HD145675        1        2        0   
4     16 Cyg B b  16 Cyg B  HD186427        3        1        0   
...          ...       ...       ...      ...      ...      ...   
5590   ups And b   ups And    HD9826        2        3        0   
5591   ups And c   ups And    HD9826        2        3        0   
5592   ups And d   ups And    HD9826        2        3        0   
5593   ups Leo b   ups Leo       NaN        1        1        0   
5594    xi Aql b    xi Aql  HD188310        1        1        0   

      discoverymethod  pl_controv_flag    pl_orbper  pl_orbpererr1  ...  \
0     Radial Velocity                0   323.210000       0.060000  ...   
1     Radial Velocity                0   516.

In [ ]:
df[(df['slope1']<0) & (df['slope2']>0) & (df['slope3']>0)].shape

(6, 5)

In [ ]:
df[(df['slope1']>0) & (df['slope2']>0) & (df['slope3']>0)].shape

(31, 5)

In [ ]:
#check with planet mass and slope
#consider all positive or negative

In [ ]:
df.describe()

           slope1      slope2      slope3  assigned_value
count  131.000000  131.000000  131.000000      131.000000
mean     0.000004   -0.000002   -0.000001       -0.022901
std      0.000076    0.000075    0.000072        1.003576
min     -0.000184   -0.000181   -0.000204       -1.000000
25%     -0.000035   -0.000054   -0.000046       -1.000000
50%      0.000004   -0.000007   -0.000002       -1.000000
75%      0.000042    0.000041    0.000039        1.000000
max      0.000220    0.000269    0.000197        1.000000

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df['assigned_value'], bins =5)

(array([67.,  0.,  0.,  0., 64.]),
 array([-1. , -0.6, -0.2,  0.2,  0.6,  1. ]),
 <BarContainer object of 5 artists>)

<Figure size 640x480 with 1 Axes>

In [ ]:
df

In [ ]:
df[(df['assigned_value'])==1]


          Star    slope1    slope2    slope3  assigned_value
2    BD-082823  0.000004 -0.000040  0.000013               1
3      HD23472  0.000058  0.000027  0.000091               1
5     HD162020 -0.000034  0.000105  0.000077               1
8     HD114386 -0.000042  0.000027  0.000068               1
9      HD40307  0.000171  0.000158  0.000018               1
..         ...       ...       ...       ...             ...
120    HD24085  0.000098  0.000167  0.000061               1
122   HD209458  0.000022  0.000046  0.000026               1
125    HD25171  0.000088  0.000034  0.000059               1
127   HD212301 -0.000048  0.000120  0.000061               1
130      HD142  0.000097  0.000269  0.000101               1

[64 rows x 5 columns]

In [ ]:
# check

In [ ]:
nn hy7

# List of elements and their Tc
vol_element_list = {
    'C' : 78,
    'O' : 183,
    'S' : 672,
    'Zn': 704,
    'Cu': 1034,
    'Na': 1035,
    'Mn': 1123,
}

ref_element_list = {
    'Cr': 1291,
    'Si': 1314,
    'Mg': 1343,
    'Co': 1354,
    'Ni': 1363,
    'V':  1370,
    'Ba': 1423,
    'Ce': 1454,
    'Ca': 1535,
    'Sc': 1541,
    'Sr': 1548,
    'Y' : 1551,
    'Ti': 1565,
    'Al': 1652,

}

def extract_abundance_diff(star_name):
    # Find row number for given star name
    row = df_f.loc[df_f['id'] == star_name]
    row_index = row.index[0]

    # Extract column values for given elements
    vol_abundance_diff1 = []
    ref_abundance_diff1 = []
    vol_abundance_diff2 = []
    ref_abundance_diff2 = []
    vol_abundance_diff3 = []
    ref_abundance_diff3 = []

    #error = []
    for element in vol_element_list.keys():
        col_name = f'1_[{element}/Fe]'
        error_col_name = f'1_e_[{element}/Fe]'
        if col_name in df_f.columns and error_col_name in df_f.columns:
            col_value = df_f.at[row_index, col_name]
            #error_value = df5.at[row_index, error_col_name]
            vol_abundance_diff1.append(col_value)
           # error.append(error_value)

    for element in ref_element_list.keys():
        col_name = f'1_[{element}/Fe]'
        error_col_name = f'1_e_[{element}/Fe]'
        if col_name in df_f.columns and error_col_name in df_f.columns:
            col_value = df_f.at[row_index, col_name]
            #error_value = df5.at[row_index, error_col_name]
            ref_abundance_diff1.append(col_value)
           # error.append(error_value)

    for element in vol_element_list.keys():
        col_name = f'2_[{element}/Fe]'
        error_col_name = f'2_e_[{element}/Fe]'
        if col_name in df_f.columns and error_col_name in df_f.columns:
            col_value = df_f.at[row_index, col_name]
            #error_value = df5.at[row_index, error_col_name]
            vol_abundance_diff2.append(col_value)
           # error.append(error_value)

    for element in ref_element_list.keys():
        col_name = f'2_[{element}/Fe]'
        error_col_name = f'2_e_[{element}/Fe]'
        if col_name in df_f.columns and error_col_name in df_f.columns:
            col_value = df_f.at[row_index, col_name]
            #error_value = df5.at[row_index, error_col_name]
            ref_abundance_diff2.append(col_value)
           # error.append(error_value)

    for element in vol_element_list.keys():
        col_name = f'3_[{element}/Fe]'
        error_col_name = f'3_e_[{element}/Fe]'
        if col_name in df_f.columns and error_col_name in df_f.columns:
            col_value = df_f.at[row_index, col_name]
            #error_value = df5.at[row_index, error_col_name]
            vol_abundance_diff3.append(col_value)
           # error.append(error_value)

    for element in ref_element_list.keys():
        col_name = f'3_[{element}/Fe]'
        error_col_name = f'3_e_[{element}/Fe]'
        if col_name in df_f.columns and error_col_name in df_f.columns:
            col_value = df_f.at[row_index, col_name]
            #error_value = df5.at[row_index, error_col_name]
            ref_abundance_diff3.append(col_value)
           # error.append(error_value)

    return vol_abundance_diff1, ref_abundance_diff1, vol_abundance_diff2, ref_abundance_diff2, vol_abundance_diff3, ref_abundance_diff3 #error]

In [ ]:
df_T_vol = df_T[df_T['Temperature']<1200]
df_T_ref = df_T[df_T['Temperature']>1200]


   Element  Temperature
1       Al         1652
3       Ti         1565
4        Y         1551
5       Sr         1548
6       Sc         1541
7       Ca         1535
9       Ce         1454
10      Ba         1423
11       V         1370
12      Ni         1363
13      Co         1354
14      Mg         1343
16      Si         1314
17      Cr         1291

In [ ]:
ext = extract_abundance_diff('HD102117')
ext

([-0.06, -0.020000000000000018, 0.0049999999999999906, -0.011999999999999997],
 [0.01999999999999999,
  -0.010000000000000002,
  -0.06,
  -0.033,
  0.006999999999999999,
  -0.052000000000000005,
  0.03,
  -0.03,
  -0.03,
  0.017,
  0.020000000000000004,
  0.06,
  -0.03,
  -0.06],
 [0.01, -0.010000000000000009, -0.09500000000000001, 0.0030000000000000027],
 [0.035,
  0.019999999999999997,
  -0.07,
  -0.016000000000000014,
  -0.005999999999999998,
  0.015,
  0.04000000000000001,
  0.07,
  0.019999999999999997,
  0.012999999999999998,
  0.09,
  0.05,
  0.03,
  0.030000000000000006],
 [0.0, 0.0, -0.06999999999999999, -0.137],
 [-0.030000000000000027,
  0.0,
  0.0,
  -0.021999999999999992,
  -0.013000000000000005,
  0.016,
  0.05000000000000002,
  -0.03,
  -0.01,
  0.015,
  0.08000000000000002,
  -0.020000000000000004,
  0.05,
  0.0])

In [ ]:
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting

In [ ]:
def fit_line(x,y):
  init = models.Linear1D()
  fit  = fitting.LinearLSQFitter()
  model = fit(init,x,y)
  return model

In [ ]:
#df = pd.DataFrame(columns=['Star','slope1','slope2','slope3'])
dat1 = []
for i,row in df_f.iterrows():
  id = row['id']

  ext = extract_abundance_diff(id)

  l0_vol = np.array(ext[0]).flatten()
  l0_ref = np.array(ext[1]).flatten()
  l1_vol = np.array(ext[2]).flatten()
  l1_ref = np.array(ext[3]).flatten()
  l2_vol = np.array(ext[4]).flatten()
  l2_ref = np.array(ext[5]).flatten()


  T_vol = df_T_vol['Temperature']
  T_ref = df_T_ref['Temperature']

  mod1  = fit_line(T_vol,l0_vol)
  x_new1 = np.linspace(T_vol.min(), T_vol.max(),100)
  y_new1 = mod1(x_new1)

  mod2  = fit_line(T_ref,l0_ref)
  x_new2 = np.linspace(T_ref.min(), T_ref.max(),100)
  y_new2 = mod1(x_new2)

  mod13 = fit_line(T_vol,l0_vol)
  x_new3 = np.linspace(T_vol.min(), T_vol.max(),100)
  y_new3 = mod1(x_new3)

  mod4  = fit_line(T_ref,l0_ref)
  x_new4 = np.linspace(T_ref.min(), T_ref.max(),100)
  y_new4 = mod1(x_new4)

  mod5  = fit_line(T_vol,l0_vol)
  x_new5 = np.linspace(T_vol.min(), T_vol.max(),100)
  y_new5 = mod1(x_new5)

  mod6  = fit_line(T_ref,l0_ref)
  x_new6 = np.linspace(T_ref.min(), T_ref.max(),100)
  y_new6 = mod1(x_new6)

  #df.append({'Star':row['Star']}, ignore_index=True)
  #df.append({'slope1':mod1.slope.value}, ignore_index=True)
  #df.append({'slope2':mod2.slope.value}, ignore_index=True)
  #df.append({'slope3':mod3.slope.value}, ignore_index=True)

  #df['slope1'] = mod1.slope.value
  #df['slope2'] = mod2.slope.value
  #df['slope3'] = mod3.slope.value

  dat.append([row['id'], mod1.slope.value, mod2.slope.value, mod3.slope.value])

  fig,ax = plt.subplots(nrows=1,ncols=3,figsize=[30,8])
  ax[0].scatter(T_vol,l0_vol)
  ax[0].plot(x_new1,y_new1,'--',c='r')
  ax[0].scatter(T_ref,l0_ref)
  ax[0].plot(x_new2,y_new2,'--',c='b')
  ax[1].scatter(T_vol,l1_vol)
  ax[1].plot(x_new3,y_new3,'--',c='r')
  ax[1].scatter(T_ref,l1_ref)
  ax[1].plot(x_new4,y_new4,'--',c='b')
  ax[2].scatter(T_vol,l1_vol)
  ax[2].plot(x_new5,y_new5,'--',c='r')
  ax[2].scatter(T_ref,l1_ref)
  ax[2].plot(x_new6,y_new6,'--',c='b')

  #ax[1].scatter(T,l1)
  #ax[1].plot(x_new2,y_new2,'--',c='r')
  #ax[2].scatter(T,l2)
  #ax[2].plot(x_new3,y_new3,'--',c='r')

  fig.suptitle(f"SwP: {row['id']}", y = 0.97, x= 0.50)






<ipython-input-73-6a6302b49940>:54: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig,ax = plt.subplots(nrows=1,ncols=3,figsize=[30,8])


<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

...

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

<Figure size 3000x800 with 3 Axes>

In [ ]:
mod1.slope.value


-2.9808125332890286e-05

In [ ]:
for i,row in df_f.iterrows():
  id = row['Star']

  ext = extract_abundance_diff(id)

  l0 = np.array(ext[0]).flatten()
  l1 = np.array(ext[1]).flatten()
  l2 = np.array(ext[2]).flatten()

  T = df_T['Temperature']

  fig,ax = plt.subplots(nrows=i,ncols=3,figsize=[5,3])
  ax[1,0].scatter(T,l0)
  ax[1,1].scatter(T,l1)
  ax[1,2].scatter(T,l2)


  if i == 5:
     break












In [ ]:
for i,row in df_f.iterrows():
  id = row['Star']

  ext = extract_abundance_diff(id)

  l0 = np.array(ext[0]).flatten()
  l1 = np.array(ext[1]).flatten()
  l2 = np.array(ext[2]).flatten()

In [ ]:
df6

   Element  Temperature
3       Ti         1565
5       Sr         1548
6       Sc         1541
7       Ca         1535
9       Ce         1454
10      Ba         1423
12      Ni         1363
13      Co         1354
14      Mg         1343
16      Si         1314
17      Cr         1291
18      Mn         1123
20      Cu         1034
21      Zn          704
22       S          672
23       O          183
24       C           78

In [ ]:
l0

array([0.08 , 0.048, 0.081, 0.03 , 0.05 , 0.   , 0.1  , 0.011, 0.04 ,
       0.01 , 0.055, 0.073, 0.016, 0.11 , 0.03 , 0.25 , 0.128, 0.04 ,
       0.13 , 0.01 , 0.12 ])

In [ ]:
columns = ['id']
ids = ['First', 'Second','Third']
for i in range(1,4):
  columns+= [ids[i-1]]
  c = [f'{i}_'+ j for j in param]
  c_err = [f'{i}_'+ j for j in param_err]

  columns += c + c_err

In [ ]:
df3 = pd.DataFrame(data_main, columns=columns)

In [ ]:
df3.to_csv('Abundance Difference.csv')

In [ ]:
df3

          id     First  1_[Fe/H]  1_[Fe/H]_a  1_[Fe/H]_s  1_[Ca/Fe]  \
0   HD103197   HD82783      0.01        0.01        0.01      -0.04   
1    HD20781  HD176157       NaN         NaN         NaN        NaN   
2    HD10700    HD1171       NaN         NaN         NaN        NaN   
3    HD51608   HD24633     -0.03       -0.03       -0.03      -0.01   
4   HD130322   HD24633      0.02        0.02        0.02      -0.03   
..       ...       ...       ...         ...         ...        ...   
82  HD108147  HD139590       NaN         NaN         NaN        NaN   
83  HD212301  HD206116     -0.06       -0.06       -0.06       0.02   
84  HD179949  HD206116     -0.03       -0.03       -0.03      -0.02   
85  HD221287   HD68287     -0.02       -0.02       -0.02       0.05   
86     HD142   HD68287      0.03        0.03        0.03      -0.25   

    1_[Sr/Fe]  1_[Sc/Fe]  1_[V/Fe]  1_[Co/Fe]  ...  3_e_[Mg/Fe]  3_e_[Ti/Fe]  \
0       -0.02     -0.056    -0.030     -0.033  ...          NaN    

In [ ]:
df4 = pd.read_csv('General Properties.csv')
df4

            Star  Teff (K)  e_Teff  logg [cm/s2]  e_logg  [Fe/H]  e_[Fe/H]  \
0       HIP38324      4393    98.0          4.01    0.39   -0.32      0.03   
1        HD85512      4400    45.0          4.36    0.10   -0.26      0.14   
2       HD192961      4418    51.0          4.23    0.18   -0.30      0.22   
3       HD218511      4424    90.0          4.52    0.26   -0.05      0.02   
4       HIP25612      4433    95.1          4.33    0.35   -0.49      0.06   
...          ...       ...     ...           ...     ...     ...       ...   
1023     HD66740      6666    39.0          4.49    0.06    0.04      0.02   
1024    HD141128      6758    48.0          4.67    0.06    0.07      0.03   
1025  CD-2310879      6788    43.0          4.67    0.04   -0.24      0.02   
1026     HD20852      6813    70.0          4.76    0.06   -0.35      0.04   
1027     HD38385      7212    83.0          4.61    0.11    0.09      0.05   

      RA (degree)  Dec (degree)  
0       117.76394     -50.305

In [ ]:
df4 = df4[['Star','Teff (K)','e_Teff','logg [cm/s2]','e_logg','RA (degree)','Dec (degree)','[Fe/H]',	'e_[Fe/H]']]
df4

            Star  Teff (K)  e_Teff  logg [cm/s2]  e_logg  RA (degree)  \
0       HIP38324      4393    98.0          4.01    0.39    117.76394   
1        HD85512      4400    45.0          4.36    0.10    147.77938   
2       HD192961      4418    51.0          4.23    0.18    304.90390   
3       HD218511      4424    90.0          4.52    0.26    347.42063   
4       HIP25612      4433    95.1          4.33    0.35     82.07632   
...          ...       ...     ...           ...     ...          ...   
1023     HD66740      6666    39.0          4.49    0.06    121.00545   
1024    HD141128      6758    48.0          4.67    0.06    236.90875   
1025  CD-2310879      6788    43.0          4.67    0.04    195.86471   
1026     HD20852      6813    70.0          4.76    0.06     50.18577   
1027     HD38385      7212    83.0          4.61    0.11     85.87577   

      Dec (degree)  [Fe/H]  e_[Fe/H]  
0        -50.30500   -0.32      0.03  
1        -43.50278   -0.26      0.14  
2     

In [ ]:
df4.to_csv('Properties_f.csv')

In [ ]:
!pip install astroML
from astroML.crossmatch import crossmatch_angular

In [ ]:
df5 = pd.read_csv('Final Dataset.csv')
df5

    col1_1      Star  Teff (K)  e_Teff  logg [cm/s2]  e_logg  RA (degree)  \
0      307  HD103197      5250    60.0          4.45    0.11    178.22074   
1      313   HD20781      5256    29.0          4.37    0.05     50.01226   
2      327   HD10700      5310    17.0          4.44    0.03     26.01701   
3      351   HD51608      5358    22.0          4.36    0.05    103.71392   
4      359  HD130322      5365    29.0          4.37    0.05    221.88635   
..     ...       ...       ...     ...           ...     ...          ...   
82     980  HD108147      6260    29.0          4.47    0.03    186.44278   
83     982  HD212301      6271    26.0          4.55    0.03    336.87883   
84     988  HD179949      6287    28.0          4.54    0.04    288.88846   
85    1007  HD221287      6374    29.0          4.62    0.03    352.83474   
86    1009     HD142      6403    65.0          4.62    0.07      1.57990   

    Dec (degree)  col1_2        id  ... 3_e_[Mg/Fe]  3_e_[Ti/Fe]  3_e_[Ba/F

In [ ]:
df5 = df5.drop(['col1_1','col1_2'], axis=1)
df5

        Star  Teff (K)  e_Teff  logg [cm/s2]  e_logg  RA (degree)  \
0   HD103197      5250    60.0          4.45    0.11    178.22074   
1    HD20781      5256    29.0          4.37    0.05     50.01226   
2    HD10700      5310    17.0          4.44    0.03     26.01701   
3    HD51608      5358    22.0          4.36    0.05    103.71392   
4   HD130322      5365    29.0          4.37    0.05    221.88635   
..       ...       ...     ...           ...     ...          ...   
82  HD108147      6260    29.0          4.47    0.03    186.44278   
83  HD212301      6271    26.0          4.55    0.03    336.87883   
84  HD179949      6287    28.0          4.54    0.04    288.88846   
85  HD221287      6374    29.0          4.62    0.03    352.83474   
86     HD142      6403    65.0          4.62    0.07      1.57990   

    Dec (degree)        id     First  1_[Fe/H]  ...  3_e_[Mg/Fe]  3_e_[Ti/Fe]  \
0      -50.29282  HD103197   HD82783      0.01  ...          NaN          NaN   
1      -2

In [ ]:
import seaborn as sb
from astropy.table import Table
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (7,7)
matplotlib.rcParams['font.size'] = 15

In [ ]:
df6 = pd.read_csv('Condensation Temps.csv')
df6

   Element  Temperature
0       Zr         1722
1       Al         1652
2       Nd         1630
3       Ti         1565
4        Y         1551
5       Sr         1548
6       Sc         1541
7       Ca         1535
8       Eu         1491
9       Ce         1454
10      Ba         1423
11       V         1370
12      Ni         1363
13      Co         1354
14      Mg         1343
15      Fe         1338
16      Si         1314
17      Cr         1291
18      Mn         1123
19      Na         1035
20      Cu         1034
21      Zn          704
22       S          672
23       O          183
24       C           78

In [ ]:
df6 = df6.drop(df6.index[[0,2,8,15]])
df6

   Element  Temperature
1       Al         1652
3       Ti         1565
4        Y         1551
5       Sr         1548
6       Sc         1541
7       Ca         1535
9       Ce         1454
10      Ba         1423
11       V         1370
12      Ni         1363
13      Co         1354
14      Mg         1343
16      Si         1314
17      Cr         1291
18      Mn         1123
19      Na         1035
20      Cu         1034
21      Zn          704
22       S          672
23       O          183
24       C           78

In [ ]:
# List of elements and their Tc
element_list = {
    'C' : 78,
    'O' : 183,
    'S' : 672,
    'Zn': 704,
    'Cu': 1034,
    'Na': 1035,
    'Mn': 1123,
    'Cr': 1291,
    'Si': 1314,
    'Mg': 1343,
    'Co': 1354,
    'Ni': 1363,
    'V':  1370,
    'Ba': 1423,
    'Ce': 1454,
    'Ca': 1535,
    'Sc': 1541,
    'Sr': 1548,
    'Y' : 1551,
    'Ti': 1565,
    'Al': 1652,

}

def extract_abundance_diff(star_name):
    # Find row number for given star name
    row = df3.loc[df3['id'] == star_name]
    row_index = row.index[0]

    # Extract column values for given elements
    abundance_diff = []
    error = []
    for element in element_list.keys():
        col_name = f'1_[{element}/Fe]'
        error_col_name = f'1_e_[{element}/Fe]'
        if col_name in df3.columns and error_col_name in df3.columns:
            col_value = df3.at[row_index, col_name]
            #error_value = df5.at[row_index, error_col_name]
            abundance_diff.append(col_value)
           # error.append(error_value)

    return [abundance_diff] #error]

In [ ]:
l=np.array(extract_abundance_diff('HD179949')[0]).flatten()


In [ ]:
extract_abundance_diff('HD179949')

[[0.021999999999999992,
  -0.33899999999999997,
  0.0049999999999999906,
  -0.009999999999999995,
  -0.12,
  -0.16,
  -0.005000000000000001,
  -0.023999999999999994,
  0.04,
  0.010000000000000009,
  -0.032,
  -0.026,
  -0.068,
  0.05,
  -0.26,
  -0.019999999999999997,
  0.005,
  0.06,
  0.03,
  -0.03,
  0.07999999999999999]]

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(df6['Temperature'],l,marker='o')
plt.axvline(1200)

<Figure size 700x700 with 1 Axes>

In [ ]:
def fit_line(x,y):
  init = models.Linear1D()
  fit  = fitting.LinearLSQFitter()
  model = fit(init,x,y)
  return model


In [ ]:
plt.scatter(df6['Temperature'],l, c='r', label = 'S1')

mod1  = fit_line(df6['Temperature'],l)
x_new = np.linspace(df6['Temperature'].min(), df6['Temperature'].max(),100)
y_new = mod1(x_new)
plt.plot(x_new,y_new, '--', label = 'S1 Fit')

<Figure size 700x700 with 1 Axes>

In [ ]:
def centroid(n,df):
  sample1 = df5[:n]
  sample2 = df5[n:]

  x1_c = sample1['Fe'].mean()
  y1_c = sample1['y'].mean()
  x2_c = sample2['x'].mean()
  y2_c = sample2['y'].mean()

  centroids =[[x1_c,y1_c],
              [x2_c,y2_c]]

  t = pd.DataFrame(centroids, columns=['x','y'])
  slope = (y2_c-y1_c)/(x2_c-x1_c)

  fig = plt.figure(figsize = (10,10))
  ax  = fig.add_subplot()

  ax.scatter(sample1['x'], sample1['y'], c='r', label = 'S1')

  mod1  = fit_line(sample1['x'],sample1['y'])
  x_new = np.linspace(sample1['x'].min(), sample1['x'].max(),100)
  y_new = mod1(x_new)

  ax.plot(x_new,y_new, '--', label = 'S1 Fit')

  ax.scatter(sample2['x'], sample2['y'],c='b', label = 'S2')

  mod2  = fit_line(sample2['x'],sample2['y'])
  x_new = np.linspace(sample2['x'].min(), sample2['x'].max(),100)
  y_new = mod2(x_new)

  ax.plot(x_new,y_new, '--', label = 'S2 Fit')

  ax.plot(t['x'],t['y'],'--o', color = 'black', label = 'Centroids')

  ax.set_xlabel('x')
  ax.set_ylabel('y')
  ax.legend()

  return slope, centroids, fig, ax

In [ ]:
elem_list = df3.columns.to_list()

In [ ]:
for val in elem_list:
  print(re.search)

In [ ]:
elems=df6['Element'].to_list()
elems_renamed = ['id','First',]
for val in elems:
  elems_renamed.append('1_['+val+'/Fe]')

In [ ]:
df3_extract = df3[elems_renamed]
df3_extract

          id     First  1_[Al/Fe]  1_[Ti/Fe]  1_[Y/Fe]  1_[Sr/Fe]  1_[Sc/Fe]  \
0   HD103197   HD82783      -0.13      -0.04     -0.01      -0.02     -0.056   
1    HD20781  HD176157        NaN        NaN       NaN        NaN        NaN   
2    HD10700    HD1171        NaN        NaN       NaN        NaN        NaN   
3    HD51608   HD24633      -0.08      -0.10     -0.12      -0.21     -0.032   
4   HD130322   HD24633      -0.14      -0.17      0.00      -0.11     -0.077   
..       ...       ...        ...        ...       ...        ...        ...   
82  HD108147  HD139590        NaN        NaN       NaN        NaN        NaN   
83  HD212301  HD206116      -0.03      -0.03      0.01       0.06     -0.002   
84  HD179949  HD206116       0.08      -0.03      0.03       0.06      0.005   
85  HD221287   HD68287      -0.04       0.01      0.10       0.07     -0.120   
86     HD142   HD68287      -0.12       0.01      0.13       0.04     -0.128   

    1_[Ca/Fe]  1_[Ce/Fe]  1_[Ba/Fe]  ..

In [ ]:
plt.figure(figsize=[10,8])
for i, row in df3_extract.iterrows():
  plt.plot(df6.Temperature,df3_extract.loc[df3_extract.id==row.id].values[0][2:],'o')
plt.axvline('1200')

<Figure size 1000x800 with 1 Axes>

In [ ]:
df6

   Element  Temperature
1       Al         1652
3       Ti         1565
4        Y         1551
5       Sr         1548
6       Sc         1541
7       Ca         1535
9       Ce         1454
10      Ba         1423
11       V         1370
12      Ni         1363
13      Co         1354
14      Mg         1343
16      Si         1314
17      Cr         1291
18      Mn         1123
19      Na         1035
20      Cu         1034
21      Zn          704
22       S          672
23       O          183
24       C           78

In [ ]:
import matplotlib.gridspec as gridspec

In [ ]:
for i,row in df_f.iterrows():

  fig= plt.figure(figsize = (33,8))

  gs=gridspec.GridSpec(1,3, width_ratios=[0.8,1,1])


  ax1 = fig.add_subplot(gs[0,0])
  ax1.imshow(optical.data)

  ax2 = fig.add_subplot(gs[0,1],projection = wcs1)
  img1=ax2.imshow(nuv, cmap ='gray_r', origin='lower')
  ax2.set_title('NUV')
  plt.colorbar(img1, shrink = 0.8)
  optical.overlay_hexagon(ax=ax2, transform=ax2.get_transform(optical.wcs)
  , color = 'black', linewidth=3)

  ax3 = fig.add_subplot(gs[0,2],projection = wcs2)
  img2=ax3.imshow(fuv, cmap ='gray_r', origin='lower')
  ax3.set_title('FUV')
  plt.colorbar(img2, shrink = 0.8)
  optical.overlay_hexagon(ax=ax3, transform=ax3.get_transform(optical.wcs)
  , color = 'black', linewidth=3)

NameError: ignored